# Lab 05-2 - $k$-nearest neighbors

In [ ]:
## Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

# Let's turn off the FutureWarning messages that are generated by some of the sklearn functions we will be using. 
# These warnings are not relevant to our work and can be safely ignored.
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  

This lab uses the results from `lab04-2` on transformations, scaling, and pipelines. Note I've copied the necessary objects below, simplifying the code and renaming some of the variables slightly as well. See the solutions for the lab for more details.

In [ ]:
# Housing data from Iowa: n = 777
ic = pd.read_csv("../data/IowaCityHomeSales.csv")

# Train test split of data
# - train n = 621
# - test n = 156
from sklearn.model_selection import train_test_split
train, test = train_test_split(ic, test_size=0.2, random_state=76)

# We'll only use the following p=3 numerical predictor variables:
example_pred_vars = ['area.lot', 'area.living', 'area.garage1']

# training: y outcome variable, x predictor variables
train_y = train['sale.amount']
train_x = train[example_pred_vars]

# test: y outcome variable, x predictor variables
test_y = test['sale.amount']
test_x = test[example_pred_vars]

# Create a transformation and scaler pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline 
my_pipeline = Pipeline(steps = [
    ('transformer', PowerTransformer(method = 'yeo-johnson')),
    ('scaler', MinMaxScaler())
])

# .fit() pipeline on training data to transform and scale variables 
fitted_pipe = my_pipeline.fit(X = train_x)

# .transform() both the training and test predictor variables
# We will use these sets of p=3 predictor variables
train_x_transformed = fitted_pipe.transform(X = train_x)
test_x_transformed = fitted_pipe.transform(X = test_x)

## A) Error Metric

If we are going to make predictions $\widehat{y}$ of a true $y$, we need a measure of error of our predictions i.e. our prediction model's performance.  We can evaluate the performance of this model by looking at the root mean squared error on the test data 

$$RMSE =  \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \widehat{y}_i)^2}$$

Note the advantage of using RMSE over MSE is that the units of RMSE match the units of the outcome variable $y$

Let's import the [`root_mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html) function from `sklearn`. Click on the link to see what the inputs are.


In [ ]:
from sklearn.metrics import root_mean_squared_error

## B) Model 0: Mean Model

Following the spirit of a Minimum Viable Product, let's start with a really simple model. The simplest model we can fit is $\widehat{y} = \overline{y}$ i.e. the mean of around \$180,000. In other words use the average house price from the training data as our prediction for all houses:

In [ ]:
train_y_mean = np.average(a = train_y)
print(train_y_mean)

Two questions arise:
1. What is the error of this prediction on the train data?
1. What is the error of this prediction on the test data?



In [ ]:
# We create an array of length the number of rows in our training
# data i.e. 621. Why? Because to use root_mean_squared_error the
# inputs y_true and y_pred need to match in length
y_hat_train_model_0 = np.repeat(a = train_y_mean, repeats = train.shape[0])

# Compute error on training data
root_mean_squared_error(y_true = train_y, y_pred = y_hat_train_model_0)


We compute the error on the test data.

In [ ]:
y_hat_test_model_0 = np.repeat(a = train_y_mean, repeats = test.shape[0])
root_mean_squared_error(y_true = test_y, y_pred = y_hat_test_model_0)

Observe it increased from around \$83,000 to \$115,000. Which makes sense since the mean model is based on training data.

## C) Model 1: $k$-nearest Neighbors

Model 0 of the mean is a simple model, but it doesn't use any of the p = 3 predictor variables. Let's use this information to hopefully yield better predictions.

One example of a more complex model would be linear regression (which we'll revisit later). Today we're going to use $k$-nearest neighbors model to predict the sale prices of homes. For:

* A regression task we use `KNeighborsRegressor()`
* A classification task we use `KNeighborsClassifier()`

Since our outcome variable $y$ is numerical, we'll use `KNeighborsRegressor()`. The code below initializes a `KNeighborsRegressor()` object using

1. $k$ = 10 neighbors. Note here since we aren't dealing with a binary categorical outcome, we can use an even number for $k$ since there isn't the possibility of ties like in the example from the lecture slides.
1. uniform weighting i.e. all $k$ = 10 points are weighted equally by $\frac{1}{10}$, regardless of the distance
1. Euclidean distance (Minkowski distance with $p=2$):

In [ ]:
# Setup knn regression object
from sklearn.neighbors import KNeighborsRegressor
knn_reg_10 = KNeighborsRegressor(n_neighbors=10, weights='uniform', p=2)

Like many of the other `sklearn` objects we've worked with, we can then fit this model to the training data. But this time we need to provide both the predictors and the outcome variable:

In [ ]:
# Fit the model on the training data: 
# - train_transformed predictor variables
# - train_y outcome variable
knn_reg_10.fit(X = train_x_transformed, y = train_y)

In this example you'll notice that we did not need to store a separate fitted model, instead the `fit()` method modifies our existing `KNeighborsRegressor` object and saves it there. That said, it's not really an issue if you wanted to store fitted model in a separate object, especially for the small-sized models we'll be using early on in this course.

Now let's:
1. Make predictions $\widehat{y}$ for the training data using `.predict()`
1. Plot them against the true $y$ for the training data
1. Compute the RMSE

In [ ]:
y_hat_train_knn_10 = knn_reg_10.predict(X = train_x_transformed)
plt.scatter(x=train_y, y = y_hat_train_knn_10)
plt.xlabel("True y (in $)")
plt.ylabel("Predicted y_hat (in $)")
plt.title("True vs kNN predicted house price for training data")
plt.show()
root_mean_squared_error(y_true=train_y, y_pred=y_hat_train_knn_10)


Observe the RMSE has dropped from around \$83,000 to \$45,000 i.e. a difference of around \$38,000. It makes sense after all, we are using the following predictor variables: 

1. `area.lot`: lot size
1. `area.living`: living area of the home
1. `area.garage1`: area of garage

Knowing this information provides better predictions of the house price

## D) Question 1

**Part a)** Re-purpose the code above to fill in the X, Y, Z, A, and B values where indicated below. Note since we set the `random_state=76` for the random train/test split, you should all in theory get the same answers.

**Part b)** After you've filled in the table answer the following "why" questions. 
1. Which is bigger? Y or B? Explain why this is the case.
1. Let's say you have to predict house prices for a new set of data where you are NOT given the true price $y$ rather, only the p = 3 predictor variables x. **Based on only the 9 values here**, which of the 3 models would you use to make predictions and why?
1. There are things you can do to improve the above analysis. Think of one and justify why. Hint: think train/test split. 



In [ ]:
# Code for Part a) 


|                         | Model 0: mean | $k$-nearest neighbors, $k=10$ | $k$-nearest neighbors, $k=1$ |
|-------------------------|--------------:|------------------------------:|-----------------------------:|
| RMSE on training data   |       \$83,089 |                     \$44,715 |                       Z = \$ |
| RMSE on test data       |      \$115,802 |                       X = \$ |                       A = \$ |
| Ratio: test ÷ training  |          1.39 |                           Y = |                          B = |

Solutions to Part b)

1. 
1. 
1. 

## E) Pipelines

Here we will build on our previous pipeline from the last assignment to 
    
- Step 1: Create a machine learning pipeline using
    1. A Yeo-Johnson normalizing transformation
    1. A Min-Max scaling
    1. **New**: `KNeighborsRegressor()` using
        - k = 3
        - inverse-distance weighting 
        - Euclidean distance by setting `p = 2`
- Step 2: Fit the pipeline to the training data, and then compute the RMSE on the test data
- Step 3: Change the pipeline to set k = 5, then repeat step 2
- Step 4: Generalize Step 3 for multiple values of `n_neighbors`: [3,5,10,15] using looping.
- Step 5: Using a scatterplot compare the RMSE for different values of `n_neighbors`, and report the best performing choice

You may ignore any overflow warnings (they simply reflect precision difficulties due to our outcome values being large numbers). *Hint*: You can set a parameter of one of the pipeline's named steps using the syntax given below.

In [ ]:
# Step 1:
# Define simple pipeline
my_pipeline = Pipeline(steps = [
    ('transformer',  PowerTransformer(method = 'yeo-johnson')),
    ('scaler', MinMaxScaler()),
    # Added this:
    ('model_fit', KNeighborsRegressor(n_neighbors = 3, weights = 'distance', p = 2))
])

# Step 2:
# .fit() the pipeline, but note we need to also pass y = train_y as an argument
my_pipeline.fit(X = train_x, y = train_y)

# .predict() the outcome variable using the test predictor variables
y_hat = my_pipeline.predict(X = test_x)

# compute the RMSE
print(root_mean_squared_error(y_true = test_y, y_pred = y_hat))

For Step 3: We use double underscore `__` to access an argument within a named step of the pipeline. We use this to change the k in our pipeline from 3 to 5:

In [ ]:
# Step 3: 
# Change the pipeline to fit knn for k = 5 using the double underscore __ operator. Notice
# - we want to change the model_fit step of the pipeline
# - add a __
# - the n_neighbors argument i.e. input
my_pipeline.set_params(model_fit__n_neighbors = 5)

# Now rerun the same code as above:
# .fit() the pipeline, but note we need to also pass y = train_y as an argument
my_pipeline.fit(X = train_x, y = train_y)

# .predict() the outcome variable using the test predictor variables
y_hat = my_pipeline.predict(X = test_x)

# compute the RMSE. Observe how it changes:
print(root_mean_squared_error(y_true = test_y, y_pred = y_hat))

In [ ]:
# Step 4: Only now do for loop
n_neighbors = [3,5,10,15]
# Save the resulting RMSE's in a float array of length 4. np.zeros() creates floats, unlike
# np.repeat(a = 0, ...) which would create integers and truncate the RMSE's
RMSE_n_neighbors = np.zeros(shape = len(n_neighbors))

for i in range(0, len(n_neighbors), 1):
    my_pipeline.set_params(model_fit__n_neighbors = n_neighbors[i])

    # .fit() the pipeline, but note we need to also pass y = train_y as an argument
    my_pipeline.fit(X = train_x, y = train_y)

    # .predict() the outcome variable using the test predictor variables
    y_hat = my_pipeline.predict(X = test_x)

    # compute the RMSE
    RMSE_n_neighbors[i] = root_mean_squared_error(y_true = test_y, y_pred = y_hat)

# Step 5: It appears k = 3 is best
plt.plot(n_neighbors, RMSE_n_neighbors, marker='o')
plt.xlabel("k: number of neighbors")
plt.ylabel("Root Mean Squared Error")
plt.show()

# Confirmed here
print("Best k is", n_neighbors[np.argmin(RMSE_n_neighbors)])

Observe how for these 4 values of $k$, the error increases.

## F) Question 2

Use the above code to:

- Step 1: Create a machine learning pipeline `q2_pipeline` using
    1. A Yeo-Johnson normalizing transformation
    1. A `StandardScaler()` scaling
    1. **New**: `KNeighborsRegressor()` using
        - k = 5
        - Uniform weighting 
        - Euclidean distance by setting `p = 2`
- Step 2: Generalize the above for multiple values of `n_neighbors`: 1 through 30 using looping, computing the RMSE on the test test. Hint: use `np.arange(start = 1, stop = 31)`
- Step 3: Using a scatterplot compare the RMSE for different values of `n_neighbors`, and report the best performing choice

In [ ]:
# Code for Question 2

# Step 1:


# Step 2:


# Step 3: 


## G) Reference for later: $k$-nearest Neighbors for Categorical Outcome Variables

Note this is for reference for use later, there is no question assigned.

Finally, you should recognize that we can perform the same basic steps for a classification task. A simple example is shown below, the major difference is the use of classification accuracy to evaluate the model. Let's artificially create a binary categorical variable $y$: was assessed price larger than the sale price

In [ ]:
# Create a binary outcome variable y: was the assessed value higher than the sale price?
train_y_binary = (train['assessed'] > train['sale.amount']).astype(int)
test_y_binary = (test['assessed'] > test['sale.amount']).astype(int)

Similar to our analysis above using a numerical outcome variable $y$, let's set up an ultra simple Model 0: Predict the most common binary outcome. Observe that it is `0` i.e. The sale price is greater than or equal to the assessed price i.e. "under-assessed"

In [ ]:
y_most_common = np.argmax(np.bincount(train_y_binary))
y_most_common

Instead of the RMSE, we compute the accuracy using `accuracy_score`

In [ ]:
from sklearn.metrics import accuracy_score

y_binary_hat_test = np.repeat(a = y_most_common, repeats = test.shape[0])
accuracy_score(y_true = test_y_binary, y_pred = y_binary_hat_test)

Now let's fit a `KNeighborsClassifier()` model using k = 11 to avoid ties. 

In [ ]:
# Set up a classifier with k=11 to avoid ties
from sklearn.neighbors import KNeighborsClassifier
knn_class = KNeighborsClassifier(n_neighbors=11, weights='uniform', p=2)

# .fit() the same model as before, but using new binary categorical variable, on the training data
knn_class.fit(X = train_x_transformed, y = train_y_binary)

# .predict() the outcome variable on the test data
y_knn_binary_hat_test = knn_class.predict(X = test_x_transformed)

# Compute accuracy score: proportion correct
accuracy_score(y_true = test_y_binary, y_pred = y_knn_binary_hat_test)

Here we can see that this KNN model performs slightly worse than always guessing "under-assessed" (or the integer label `0`) for every home in the test data!